In [2]:
pwd

'C:\\Users\\ajayi\\JupyterProjects\\python notebooks'

In [13]:
cd ~

C:\Users\ajayi


In [5]:
cd Desktop

C:\Users\ajayi\Desktop


In [7]:
cd python_data_analysis

C:\Users\ajayi\Desktop\python_data_analysis


In [ ]:
import python
import pandas as pd        # data handling
import numpy as np         # numerical operations
import scipy.stats as stats # inferential statistics
import matplotlib.pyplot as plt  # plots

In [ ]:
data = pd.read_excel("Mass_screening_agbado_april2026_analyze.xlsx")
 

In [ ]:
data.head() 
data.info()
data.describe()

In [ ]:
# 1. Automatically grab all categorical columns
cat_cols = data.select_dtypes(include=['object', 'category']).columns
# 2. Loop through them and print counts and percentages
for col in cat_cols:
    print(f"--- {col} ---")
    
    # Calculate counts
    counts = data[col].value_counts()
    
    # Calculate percentages (normalize=True)
    percents = data[col].value_counts(normalize=True) * 100
    
    # Combine them into a small table for readability
    df_summary = pd.DataFrame({'Count': counts, 'Percentage (%)': percents})
    print(df_summary)
    print("\n") # Adds a space between columns

In [ ]:
from scipy.stats import chi2_contingency
import pandas as pd

# 1. Define your target variable and a list of factors you want to test
target = 'chlorhexidine_gel_only'
factors = ['mother_age_cat', 'parity_cat','mother_hlfe', 'mother_occupationL', 'mother_marital_status', 'parents_religion', 'residenceL', 'place_of_anc_2_cat', 'adequate_anc', 'had_health_talk_on_ucc', 'place_of_birth_2cat', 'Healthcareworkers_source2cat', 'self_decision_2cat'] # Add all your column names here

# 2. Loop through each factor
for factor in factors:
    print(f"\n" + "="*30)
    print(f"ANALYSIS: {factor} vs {target}")
    print("="*30)
    
    # Create the frequency table (Counts)
    ct_counts = pd.crosstab(data[factor], data[target])
    
    # Create the percentage table (Row-wise percentages)
    ct_percent = pd.crosstab(data[factor], data[target], normalize='index') * 100
    
    # Perform the Chi-Square Test
    chi2, p, dof, expected = chi2_contingency(ct_counts)
    
    # Display Results
    print("\n--- Frequency Count Table (Counts) ---")
    print(ct_counts)
    print("--- Percentage Table (Row %) ---")
    print(ct_percent.round(2))
    
    print("\n--- Chi-Square Test Results ---")
    print(f"Chi-Square Statistic: {chi2:.4f}")
    print(f"P-value: {p:.4f}")
    print(f"Degrees of Freedom: {dof}")
    
    # Quick Interpretation
    alpha = 0.05
    if p < alpha:
        print("\nResult: Statistically significant (Reject H0)")
    else:
        print("\nResult: Not statistically significant (Fail to reject H0)")

In [ ]:
# for logistic regression for a categorical age column

import pandas as pd
import statsmodels.api as sm
import numpy as np

# 1. LOAD DATA
# Make sure the file name matches your edited Excel file
df = pd.read_excel("Mass_screening_agbado_april2026_analyze.xlsx")

# 2. PRE-PROCESSING
# Merge younger groups to ensure we have 'events' in the reference category
df['Age_Fixed'] = df['Age'].replace({'18-29': '18-39', '30-39': '18-39'})

# Ensure target is float (1 = High SBP, 0 = Normal)
y = df['rbg_2llr'].astype(float)

# Define all independent variables for the Adjusted model
covariates = [
    'BMI', 'GenderLR', 'SBP', 'DBP', 'Family_history_of_DMLR', 
     'tobaccoLR', 'alcoholLR', 'Age_Fixed'
]

# 3. ANALYSIS FUNCTION
def run_logit(cols):
    # drop_first=True makes the first alphabetical category the reference group
    X = pd.get_dummies(df[cols], drop_first=True, dtype=int)
    X = sm.add_constant(X)
    return sm.Logit(y, X).fit(method='bfgs', disp=False)

# 4. BUILD RESULTS TABLE
final_results = []

# Get Adjusted stats (all variables)
adj_model = run_logit(covariates)
a_ors, a_cis, a_ps = np.exp(adj_model.params), np.exp(adj_model.conf_int()), adj_model.pvalues

# Get Crude stats (one variable at a time)
for var in covariates:
    c_model = run_logit([var])
    c_ors, c_cis, c_ps = np.exp(c_model.params), np.exp(c_model.conf_int()), c_model.pvalues
    
    for i in range(1, len(c_ors)): # Skip constant
        v_name = c_ors.index[i]
        
        final_results.append({
            'Variable': v_name,
            'Crude OR (95% CI)': f"{c_ors[i]:.2f} ({c_cis.iloc[i,0]:.2f}-{c_cis.iloc[i,1]:.2f})",
            'p-value (Crude)': f"{c_ps[i]:.3f}",
            'Adjusted OR (95% CI)': f"{a_ors[v_name]:.2f} ({a_cis.loc[v_name,0]:.2f}-{a_cis.loc[v_name,1]:.2f})",
            'p-value (Adjusted)': f"{a_ps[v_name]:.3f}"
        })

# 5. DISPLAY
output_table = pd.DataFrame(final_results)
print("\n--- FINAL PUBLICATION TABLE ---")
print(output_table.to_string(index=False))

# Optional: Save to Excel for easy copy-pasting into Word
# output_table.to_excel('Regression_Results.xlsx', index=False)